## 2238 - Number of Times a Driver Was a Passenger
Table: Rides
### Table: Rides

| Column Name  | Type |
|--------------|------|
| ride_id      | int  |
| driver_id    | int  |
| passenger_id | int  |

ride_id is the primary key for this table.  
Each row of this table contains the ID of the driver and the ID of the passenger that rode in ride_id.  
Note that driver_id != passenger_id.

Write an SQL query to report the ID of each driver and the number of times they were a passenger.

Return the result table in any order.

### Example 1:

#### Input: Rides table

| ride_id | driver_id | passenger_id |
|---------|-----------|--------------|
| 1       | 7         | 1            |
| 2       | 7         | 2            |
| 3       | 11        | 1            |
| 4       | 11        | 7            |
| 5       | 11        | 7            |
| 6       | 11        | 3            |

#### Output:

| driver_id | cnt |
|-----------|-----|
| 7         | 2   |
| 11        | 0   |

### Explanation:

There are two drivers in all the given rides: 7 and 11.  
The driver with ID = 7 was a passenger two times.  
The driver with ID = 11 was never a passenger.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType
from pyspark.sql.functions import col, count, lit

spark = SparkSession.builder.getOrCreate()

# Define schema
schema = StructType([
    StructField("ride_id", IntegerType(), True),
    StructField("driver_id", IntegerType(), True),
    StructField("passenger_id", IntegerType(), True)
])

# Sample data
data = [
    (1, 7, 1),
    (2, 7, 2),
    (3, 11, 1),
    (4, 11, 7),
    (5, 11, 7),
    (6, 11, 3)
]

# Create DataFrame
df = spark.createDataFrame(data, schema)
df.createOrReplaceTempView("Rides")


In [0]:
%sql
with cte as (
  Select count(passenger_id) as cnt, passenger_id from Rides group by passenger_id
)
select distinct driver_id , coalesce(cnt ,0) from rides r left join cte c on  r.driver_id = c.passenger_id


In [0]:
from pyspark.sql.functions import *
cte = df.groupBy(col("passenger_id")).agg(count(col("passenger_id")).alias("cnt")).selectExpr("passenger_id as cte_passenger_id" , "cnt")

df.join(cte  ,col("cte_passenger_id") == col("driver_id"),"left").selectExpr("driver_id","coalesce(cnt,0)").distinct().display()

In [0]:

# SQL logic
spark.sql("""
WITH drivers AS (
    SELECT DISTINCT driver_id FROM Rides
),
passenger_counts AS (
    SELECT passenger_id, COUNT(*) AS cnt
    FROM Rides
    GROUP BY passenger_id
)
SELECT 
    d.driver_id,
    COALESCE(p.cnt, 0) AS cnt
FROM drivers d
LEFT JOIN passenger_counts p
ON d.driver_id = p.passenger_id
""").createOrReplaceTempView("Result")

# Display result
display(spark.sql("SELECT * FROM Result"))